# 1. Collecte et nettoyage des données

Une section par source. Les fichiers bruts sont téléchargés une seule fois dans `data/raw/`, les tables propres sont enregistrées dans `data/clean/`.

In [ ]:
import os
import sys

import geopandas as gpd
import pandas as pd
import yaml

# on se place à la racine du projet
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.append(os.getcwd())

# paramètres communs du projet
with open("config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)


## Exploration des contours (cartiflette)

Millésime 2026 disponible dans cartiflette (vérifié le 24/11/2026).
Fichiers bruts : communes (34 875), départements (101), régions (18), France entière, DOM compris.

In [ ]:
from src.collecte import get_contours

# récupérer les données des communes de cartiflette (téléchargées par get_contours)
annee = config["annee_geographie"]
communes = get_contours("COMMUNE", annee)

print(communes.shape)
communes.head(1)

In [ ]:
# permet de donner un résumé des variables que l'on a avec leur type et les potentielles données manquantes
resume = communes.dtypes.to_frame("type")
resume["manquants"] = communes.isna().sum()
resume

,type,manquants
ID,str,0
NOM,str,0
INSEE_COM,str,0
STATUT,str,0
POPULATION,int32,0
AREA,str,0
CTCD,str,0
EPCI,str,0
EPT,str,0
ARR,str,0


In [ ]:
# 1. chaque code commune n'apparaît qu'une fois ?
print("codes en double :", communes["INSEE_COM"].duplicated().sum())

# 2. tous les codes font bien 5 caractères ?
print("longueurs des codes :", communes["INSEE_COM"].str.len().unique())

# 3. combien de communes dans les DOM (départements qui commencent par 97) ?
dom = communes["INSEE_DEP"].str.startswith("97")
print("communes des DOM :", dom.sum(), "| métropole :", (~dom).sum())

# 4. Paris, Lyon, Marseille : une seule commune ou des arrondissements ?
print(communes.loc[communes["NOM"].isin(["Paris", "Lyon", "Marseille"]), ["INSEE_COM", "NOM"]])

codes en double : 0
longueurs des codes : [5]
communes des DOM : 129 | métropole : 34746
      INSEE_COM        NOM
129       75056      Paris
32875     69123       Lyon
34150     13055  Marseille


### Constats

Contours comporte 34 875 communes, avec 22 colonnes, il n'y a pas de valeur manquante. `INSEE_COM` pourra peut-être être notre clé de jointure (?). 129 communes sont dans les DOM (département commençant par 97), on garde les 34 746 communes de métropole. Paris, Lyon et Marseille sont des communes à elles seules. 
On garde 5 colonnes: `INSEE_COM`, `NOM`, `INSEE_DEP`, `INSEE_REG`, `geometry`. La population et la surface viendront dans les séries historiques de l'INSEE. 

Suite à cela, lors du nettoyage, les colonnes seront renommées pour être standardisées.

In [ ]:
from src.nettoyage import clean_contours

communes_propres = clean_contours(communes)   # base de données clean
print(communes_propres.shape)   # écrit la taille de la base de données
communes_propres.head(1)   #vérification avec une variable

(34746, 5)


,code_commune,nom_commune,code_departement,code_region,geometry
0,75056,Paris,75,11,"POLYGON ((2.35198 48.90148, 2.32584 48.90095, ..."


In [ ]:
# sauvegarde des contours propres
communes_propres.to_parquet("data/clean/contours_communes.parquet")

# court bilan
print("lignes :", len(communes_propres))    #nombre d'observations
print("colonnes :", communes_propres.columns.tolist())    #nombres de variables
print("valeurs manquantes :", communes_propres.isna().sum().sum())    #s'assure que sur chaque colonne il n'y a pas de valeur manquante
print("projection : EPSG", communes_propres.crs.to_epsg())    #permet de controler les valeurs et les unités de nos variables

lignes : 34746
colonnes : ['code_commune', 'nom_commune', 'code_departement', 'code_region', 'geometry']
valeurs manquantes : 0
projection : EPSG 4326


### Bilan

Le fichier propre se situe à l'adresse : `data/clean/contours_communes.parquet`
- 34 746 communes de France métropolitaine (Corse comprise), millésime 2026 ;
- 5 colonnes : `code_commune`, `nom_commune`, `code_departement`, `code_region`, `geometry` ;
- aucune valeur manquante, code commune unique sur 5 caractères ;
- coordonnées en WGS84 (EPSG 4326), à convertir en Lambert 93 (EPSG 2154) pour les distances.

## Séries historiques de population (INSEE)

Le RP 2023 publie les populations de 2017 et 2023 à la géographie du 1er janvier 2026, ainsi que la superficie et l'état civil.

### Collecte

L'instantané est archivé dans `data/raw`. La collecte suivante relit le cache.

In [ ]:
# Lire les séries INSEE brutes ; les premières lignes ne sont pas encore filtrées.
from src.collecte import get_population

population_brut = get_population(config)
population_brut.head(3)

INSEE population : 3,049,137 lignes brutes lues ; archive et empreinte contrôlées.


,GEO,GEO_OBJECT,FREQ,RP_MEASURE,OCS,OBS_STATUS,UNIT_MEASURE,TIME_PERIOD,OBS_VALUE
0,001,AAV2020,A,DWELLINGS,DW_MAIN,A,NR,1968,3446694.0
1,001,AAV2020,A,DWELLINGS,DW_SEC_DW_OCC,A,NR,1968,156421.0
2,001,AAV2020,A,DWELLINGS,DW_VAC,A,NR,1968,175385.0


### Nettoyage

On sélectionne les mesures utiles et leurs dates avant de pivoter. Les statuts INSEE de regroupement restent visibles. Ils permettent de repérer les comparaisons communales fragiles.

In [ ]:
# Garder les communes et mesures utiles, puis réunir les années dans une ligne.
from src.nettoyage import clean_population

population = clean_population(population_brut, config)
display(population.head())

Population propre : 34,746 communes métropolitaines, sans double compte des arrondissements.
  2017 : 0 valeurs absentes, 0 alertes de périmètre ; somme publiée 64,639,133.
  2023 : 0 valeurs absentes, 0 alertes de périmètre ; somme publiée 66,165,815.


,code_commune,pop_2017,statut_pop_2017,alerte_population_2017,pop_2023,statut_pop_2023,alerte_population_2023,superficie_km2,naissances_2017_2022,deces_2017_2022
0,01001,776.0,A,False,860.0,A,False,15.95,52.0,34.0
1,01002,248.0,A,False,270.0,A,False,9.15,12.0,11.0
2,01004,14035.0,A,False,15934.0,A,False,24.60,1336.0,822.0
3,01005,1689.0,A,False,1906.0,A,False,15.92,157.0,59.0
4,01006,111.0,A,False,115.0,A,False,5.88,4.0,6.0


### Contrôle et export

La clé ou le couple de clés identifie chaque observation. Le fichier propre garde ces types pour les jointures.

In [ ]:
# Vérifier qu'il reste exactement une ligne par commune.
assert population["code_commune"].is_unique
population[["pop_2017", "pop_2023"]].sum()

pop_2017    64639133.0
pop_2023    66165815.0
dtype: float64

In [ ]:
# Enregistrer la population nettoyée pour les autres étapes du projet.
population.to_parquet("data/clean/population.parquet", index=False)

## Transport principal domicile–travail (INSEE)

On sélectionne les actifs occupés de 15 ans ou plus, tous lieux de travail confondus. Les codes des modalités figurent dans config.yaml.

### Collecte

L'instantané est archivé dans `data/raw`. La collecte suivante relit le cache.

In [ ]:
# Lire les navettes INSEE brutes ; l'aperçu sert à voir les codes des modes.
from src.collecte import get_navettes

navettes_brut = get_navettes(config)
navettes_brut.head(3)

INSEE navettes : 1,587,222 lignes brutes lues ; archive et empreinte contrôlées.


,GEO,GEO_OBJECT,TRANS,AGE,EMPSTA_ENQ,WORK_AREA,RP_MEASURE,FREQ,OBS_STATUS,TIME_PERIOD,OBS_VALUE
0,001,AAV2020,1,Y_GE15,1,_T,POP,A,A,2012,220847.09555
1,001,AAV2020,2,Y_GE15,1,_T,POP,A,A,2012,416028.62459
2,001,AAV2020,3T4,Y_GE15,1,_T,POP,A,A,2012,255875.89971


### Nettoyage

Le vélo et les deux-roues motorisés forment une famille commune. Les cinq modes sont rapportés à l'ensemble des actifs occupés, y compris les actifs sans déplacement. Le seuil de 1 000 actifs intervient ensuite dans l'assemblage.

In [ ]:
# Garder les deux années et réunir les modes de déplacement comme pr?vu.
from src.nettoyage import clean_navettes

navettes = clean_navettes(navettes_brut, config)
display(navettes.head())

  Navettes 2017 : écart maximal somme des cinq modes / total = 3e-05 ; 12 alertes.
  Navettes 2023 : écart maximal somme des cinq modes / total = 3e-05 ; 0 alertes.
Navettes propres : 34,746 communes métropolitaines ; zéro publié conservé, absence conservée.


,code_commune,actifs_2017,actifs_sans_deplacement_2017,actifs_marche_2017,actifs_deux_roues_2017,actifs_voiture_2017,actifs_transports_commun_2017,alerte_navettes_2017,statut_navettes_2017,actifs_2023,actifs_sans_deplacement_2023,actifs_marche_2023,actifs_deux_roues_2023,actifs_voiture_2023,actifs_transports_commun_2023,alerte_navettes_2023,statut_navettes_2023
0,01001,346.89523,20.29790,4.04315,5.02641,305.47913,12.04865,False,A,408.42616,21.01355,7.00671,5.94218,370.45270,4.01103,False,A
1,01002,116.34568,4.08230,0.00000,0.00000,106.13992,6.12346,False,A,132.60283,11.83069,0.00000,0.97597,112.78632,7.00985,False,A
2,01004,5869.51452,241.20854,424.28783,230.48752,4169.56426,803.96637,False,A,6859.89560,257.35396,550.62490,200.31616,4876.24948,975.35110,False,A
3,01005,807.58408,39.80522,27.16817,15.55164,696.97219,28.08686,False,A,943.78774,43.58785,20.00484,8.48684,843.42729,28.28092,False,A
4,01006,50.45455,3.02727,0.00000,0.00000,46.41818,1.00909,False,A,47.41228,2.01754,0.00000,2.01754,43.37719,0.00000,False,A


### Contrôle et export

La clé ou le couple de clés identifie chaque observation. Le fichier propre garde ces types pour les jointures.

In [ ]:
# Vérifier qu'il reste une ligne par commune.
assert navettes["code_commune"].is_unique
navettes.filter(regex="^actifs_2023$|^actifs_voiture_2023$").sum()

actifs_2023            2.802871e+07
actifs_voiture_2023    1.930134e+07
dtype: float64

In [ ]:
# Enregistrer les effectifs nettoyés ; les parts seront calculées après l'assemblage.
navettes.to_parquet("data/clean/navettes.parquet", index=False)

## Exploration des jeu de données SNCF

Ce notebook traite de la collecte, l'exploration et le nettoyage des jeux de données *frequentation-gares.csv* et *gares-de-voyageurs.csv*

### 1. Gares de voyageurs

#### 1.1 Fichier brut
Les données relative aux gares, à leur identification et à leur localisation et ont été collectées au format csv auprès de la plateforme Open Data SNCF (ressources.data.sncf.com)

In [ ]:
# On récupère le fichier brut en appelant la fonction get_gares depuis src.collecte
from src.collecte import get_gares

gares = get_gares()

# On affiche les dimensions du fichier, les premières lignes et les types des variables
print(gares.shape)
display(gares.head())
resume = gares.dtypes.to_frame("type")
display(resume)

(2792, 7)


,Nom_Gare,Trigramme,Segment(s) DRG,Position géographique,Code commune,Code_UIC,Id_Gare
0,Abancourt,ABT,C,"49.6852237, 1.7743058",60001,87313759,a99c4503-7fa0-4176-b908-74f5e50cba32
1,Abbaretz,AAR,C,"47.5546432, -1.5244159",44001,87481614,5fe4e77c-2519-413d-a242-4a525e656a09
2,Abbeville,ABB,B,"50.10221, 1.82449",80001,87317362,c4a3e70f-d42c-4026-98e6-35089225688a
3,Ablon-sur-Seine,ABL,B,"48.725468, 2.419151",94001,87545269,96392cdd-5664-46c7-bd18-780813f01d2c
4,Achères Grand Cormier,GCR,B,"48.9551835, 2.0919031",78551,87386052,4f9a3fda-8e53-4286-963c-e2c8a16cb329


,type
Nom_Gare,str
Trigramme,str
Segment(s) DRG,str
Position géographique,str
Code commune,str
Code_UIC,str
Id_Gare,str


### 1.2 Constat

Dimensions : Le fichier est constitué de 2792 lignes et de 7 colonnes. 
Granularité des données : une observation par gare et par année
Clé de jointures possibles : Code_UIC pour joindre les deux bases SNCF et Code commune pour joindre aux autres bases et visualisation géographique
Colonnes à supprimer : 

In [ ]:
# On récupère le fichier nettoyé en appelant la fonction clean_gares depuis src.nettoyage
from src.nettoyage import clean_gares

gares = clean_gares()

# On affiche les dimensions du fichier, les premières lignes et les types des variables
print(gares.shape)
display(gares.head())
resume = gares.dtypes.to_frame("type")
display(resume)

(2792, 3)


,position_geographique,code_commune,code_uic
0,"49.6852237, 1.7743058",60001,87313759
1,"47.5546432, -1.5244159",44001,87481614
2,"50.10221, 1.82449",80001,87317362
3,"48.725468, 2.419151",94001,87545269
4,"48.9551835, 2.0919031",78551,87386052


,type
position_geographique,str
code_commune,str
code_uic,str


In [ ]:
# On récupère le fichier brut en appelant la fonction get_frequentation depuis src.collecte
from src.collecte import get_frequentation

frequentation = get_frequentation()

# On affiche les dimensions du fichier, les premières lignes et les types des variables
print(frequentation.shape)
display(frequentation.head())
resume = frequentation.dtypes.to_frame("type")

# On récupère le fichier nettoyé en appelant la fonction clean_frequentation depuis src.nettoyage
from src.nettoyage import clean_frequentation

frequentation = clean_frequentation()

# On affiche les dimensions du fichier, les premières lignes et les types des variables
print(frequentation.shape)
display(frequentation.head())
resume = frequentation.dtypes.to_frame("type")
display(resume)

(3010, 22)


,Nom de la gare,Code UIC,Code postal,Segmentation DRG,Total Voyageurs 2023,Total Voyageurs + Non voyageurs 2023,Total Voyageurs 2022,Total Voyageurs + Non voyageurs 2022,Total Voyageurs 2021,Total Voyageurs + Non voyageurs 2021,...,Total Voyageurs 2019,Total Voyageurs + Non voyageurs 2019,Total Voyageurs 2018,Total Voyageurs + Non voyageurs 2018,Total Voyageurs 2017,Total Voyageurs + Non voyageurs 2017,Total Voyageurs 2016,Total Voyageurs + Non voyageurs 2016,Total Voyageurs 2015,Total Voyageurs + Non voyageurs 2015
0,Abbaretz,87481614,44170,C,54862,54862,40825,40825,27466,27466,...,38473,38473,38027,38027,35637,35637,33021,33021,35989,35989
1,Achères Grand Cormier,87386052,78100,B,188426,188426,177092,177092,209673,209673,...,221003,221003,228032,228032,235698,235698,238687,238687,241046,241046
2,Achiet-le-Grand,87342048,62121,C,82597,82597,80648,80648,71432,71432,...,97848,97848,92323,92323,96451,96451,92084,92084,92018,92018
3,Agay,87757559,83530,C,39364,39364,32800,32800,25094,25094,...,22284,22284,15093,15093,14154,14154,19240,19240,19121,19121
4,Aigues-Mortes,87775858,30220,C,17034,17034,11325,11325,8201,8201,...,6146,6146,5311,5311,4822,4822,3546,3546,3917,3917


(21070, 6)


,nom_gare,code_uic,code_postal,segmentation_drg,annee,frequentation
0,Abbaretz,87481614,44170,C,2023,54862
1,Achères Grand Cormier,87386052,78100,B,2023,188426
2,Achiet-le-Grand,87342048,62121,C,2023,82597
3,Agay,87757559,83530,C,2023,39364
4,Aigues-Mortes,87775858,30220,C,2023,17034


,type
nom_gare,str
code_uic,str
code_postal,str
segmentation_drg,str
annee,int64
frequentation,int64


In [ ]:
frequentation.info()

<class 'pandas.DataFrame'>
RangeIndex: 21070 entries, 0 to 21069
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   nom_gare          21070 non-null  str  
 1   code_uic          21070 non-null  str  
 2   code_postal       21070 non-null  str  
 3   segmentation_drg  20601 non-null  str  
 4   annee             21070 non-null  int64
 5   frequentation     21070 non-null  int64
dtypes: int64(2), str(4)
memory usage: 1.5 MB


Les seules valeurs manquantes se trouvent dans la colonne segmentation_drg, celles-ci concernent 469 lignes i.e. 67 gares (469/ 7 années)

## Exploration du parc automobile (SDES)

Source: SDES, parc de véhicules routiers au niveau communal (data.gouv.fr, Licence Ouverte). 
Ce qui nous intéresse ici est de récupérer les données liées aux voitures particulières (VP) des particuliers, au 1er janvier 2017 et au 1er janvier 2023. Les voitures de société et de location sont exclues (elles sont immatriculées à l'adresse de l'entreprise ce qui sous-estimerait ou sur-estimerait le parc automobile de certaines communes).

In [ ]:
from src.collecte import get_parc_auto

parc = get_parc_auto(config["urls"]["parc_auto"], config["annee_debut"], config["annee_fin"])

print(parc.shape)
print(parc.dtypes)
parc.head()

(523325, 6)
COMMUNE_CODE      str
COMMUNE_NOM       str
CARBURANT         str
CRIT_AIR          str
PARC_2017       int64
PARC_2023       int64
dtype: object


,COMMUNE_CODE,COMMUNE_NOM,CARBURANT,CRIT_AIR,PARC_2017,PARC_2023
0,00000,Inconnu,Essence HNR,Crit'Air 1,1,143
1,00000,Inconnu,Gaz,Crit'Air 1,3,3
2,00000,Inconnu,Diesel,Crit'Air 2,307,518
3,00000,Inconnu,Diesel,Crit'Air 3,981,78
4,00000,Inconnu,Diesel,Crit'Air 4,44,28


In [ ]:
# 1.on veut compter le nombre de commune pour pouvoir les regrouper en une seule ligne (une ligne par commune)
print("communes différentes :", parc["COMMUNE_CODE"].nunique())

# 2. on compte les voitures sans commune qui vont être retirées
inconnu = parc["COMMUNE_CODE"].str.endswith("000")
for annee in ["PARC_2017", "PARC_2023"]:
    part = parc.loc[inconnu, annee].sum() / parc[annee].sum() * 100
    print(f"{annee} : {part:.2f} % des voitures ont un code inconnu")

# 3. voir si on peut trouver les arrondissements parisiens, marsaillais et lyonnais à partir des codes INSEE
arrondissements = parc["COMMUNE_CODE"].str.match(r"^(751|6938|132)")
print("codes d'arrondissements :", sorted(parc.loc[arrondissements, "COMMUNE_CODE"].unique())[:5], "...")

# 4. valeurs manquantes et total national
print("manquants :", parc.isna().sum().to_dict())
print("voitures 2017 :", parc["PARC_2017"].sum(), "| 2023 :", parc["PARC_2023"].sum())

communes différentes : 35017
PARC_2017 : 0.02 % des voitures ont un code inconnu
PARC_2023 : 0.05 % des voitures ont un code inconnu
codes d'arrondissements : ['13201', '13202', '13203', '13204', '13205'] ...
manquants : {'COMMUNE_CODE': 0, 'COMMUNE_NOM': 48, 'CARBURANT': 0, 'CRIT_AIR': 0, 'PARC_2017': 0, 'PARC_2023': 0}
voitures 2017 : 35122936 | 2023 : 35859860


### Constats

Pour les communes, on a 523 000 lignes pour 35 017 codes. Ainsi, pour avoir le nombre de voitures d'une commune, il faut additionner toutes ses lignes.
Les voitures au code de commune inconnu correspond à 0.05% du total au maximum. C'est négligeable. On peut donc se permettre de les retirer, lors du nettoyage.
En terme de parc automobile: 35,1 millions de voitures en 2017, 35,9 millions en 2023 (+2 %).
  L'ordre de grandeur est cohérent pour les voitures des particuliers.
- **À vérifier** : 35 017 codes contre 34 746 communes en métropole. L'écart vient des
  arrondissements, des DOM, des codes inconnus, et peut-être d'anciennes communes fusionnées
  (le SDES ne précise pas le millésime de ses codes communes).

### Ce que fera le nettoyage (`clean_parc_auto`)

1. retirer les codes inconnus ;
2. ramener les arrondissements à leur ville ;
3. additionner, par commune, toutes les lignes pour 2017 et pour 2023 ;
4. garder la métropole ;
5. traiter les anciens codes de communes, s'il y en a (voir la vérification suivante) ;
6. vérifier par des `assert` : code commune unique, sur 5 caractères.

In [ ]:
import pandas as pd

# on modifie notre base de données pour que Paris, Marseille et Lyon devienne des communes à part entière. 
codes = parc["COMMUNE_CODE"].copy()
codes[codes.str.startswith("751")] = "75056"   #    Paris
codes[codes.str.startswith("6938")] = "69123"   # Lyon
codes[codes.str.startswith("132")] = "13055"   #   Marseille

# voitures 2023 par commune, sans les codes inconnus
voitures = parc[~inconnu].assign(code=codes).groupby("code")["PARC_2023"].sum()

# communes des contours (métropole, 2026), avec leur nom
contours = pd.read_parquet("data/clean/contours_communes.parquet",
                           columns=["code_commune", "nom_commune"])

# 1. codes du parc absents des contours
absents = voitures[~voitures.index.isin(contours["code_commune"])]
dom = absents.index.str.startswith("97")
print("codes du parc absents des contours :", len(absents), "dont DOM :", dom.sum())
print(f"part des voitures 2023 hors DOM et absentes des contours : {absents[~dom].sum() / voitures.sum() * 100:.2f} %")
print("exemples hors DOM :", absents[~dom].index[:15].tolist())

# 2. communes des contours absentes du parc
sans_voiture = contours[~contours["code_commune"].isin(voitures.index)]
print("communes des contours absentes du parc :", len(sans_voiture))
sans_voiture.head(20)

codes du parc absents des contours : 134 dont DOM : 134
part des voitures 2023 hors DOM et absentes des contours : 0.00 %
exemples hors DOM : []
communes des contours absentes du parc : 4


,code_commune,nom_commune
16028,55050,Bezonvaux
16043,55139,Cumières-le-Mort-Homme
16062,55239,Haumont-près-Samogneux
16071,55307,Louvemont-Côte-du-Poivre


- **Correspondance avec les contours** : hors DOM, tous les codes du parc existent dans les
  contours 2026 (0,00 % des voitures perdues).
- **4 communes des contours n'ont aucune voiture dans le parc** (Bezonvaux, Cumières-le-Mort-Homme,
  Haumont-près-Samogneux, Louvemont-Côte-du-Poivre, dans la Meuse). On va donc les retirer de la base, car elles sont négligeables par rapport au reste de la base. 

In [ ]:
from src.nettoyage import clean_parc_auto

voitures = clean_parc_auto(parc, 2017, 2023)
print(voitures.shape)
print(voitures[["voitures_2017", "voitures_2023"]].sum())
voitures[voitures["code_commune"].isin(["75056", "69123", "13055"])]

(34742, 3)
voitures_2017    34350825
voitures_2023    35035890
dtype: int64


,code_commune,voitures_2017,voitures_2023
4338,13055,338308,324370
27035,69123,173356,159615
29190,75056,462773,398215


### Bilan

Fichier propre : `data/clean/parc_auto.parquet`
On remarque q'il y a en premier constat une baisse du nombre de voitures à Paris, Lyon et Marseille. 

In [ ]:
voitures.to_parquet("data/clean/parc_auto.parquet")
print("enregistré dans :", os.getcwd())

enregistré dans : /home/onyxia/work/le-rail-suit-il-la-France-qui-grandit-


## Exploration du trafic aérien (DGAC, série PAP)

Source : DGAC, trafic aérien commercial mensuel par paire d'aéroports (data.gouv.fr, Licence Ouverte v2).
Années retenues : 2017 et 2023, comme le reste du projet.

In [ ]:
from src.collecte import get_dgac

pap = get_dgac(config["urls"]["dgac_pap"], [config["annee_debut"], config["annee_fin"]])

print(pap.shape)
pap.head(1)

(3572, 13)


,ANMOIS,PAP,PAP1_LABEL,PAP2_LABEL,PAP_DIST,PAP_PEQ,PAP_PAX,PAP_FRP,PAP_DRT,PAP_MVT,PAP_PEQ_FS,PAP_PAX_FS,PAP_FRP_FS
0,201701,FMEE_FIMP,LA REUNION-ROLAND GARROS,MAURICE INTL,232,80196,77169,302.7,462,462,81134,78056,307.7


### Collecte

Les données viennent de la DGAC, publiées sur data.gouv.fr. La fonction `get_dgac` télécharge
l'archive une seule fois dans `data/raw/`, sans jamais la modifier, puis lit seulement 2017 et 2023.
Le projet peut ainsi être relancé par n'importe qui et donne toujours les mêmes données.

Nous utilisons la série PAP (trafic par paire d'aéroports) : c'est la seule qui donne les passagers
par liaison, ce qui permet de comparer chaque ligne aérienne avec le train.

In [ ]:
# 1. types et valeurs manquantes de chaque colonne
resume = pap.dtypes.to_frame("type")
resume["manquants"] = pap.isna().sum()
print(resume)

# 2. liaisons intérieures à la métropole : les deux codes OACI commencent par LF
interieur = pap["PAP"].str.match(r"^LF.._LF..$")
print("lignes intérieures à la métropole :", interieur.sum(), "sur", len(pap))

# 3. une même paire apparaît-elle dans les deux sens ? exemple Orly - Nice
print(pap.loc[pap["PAP"].isin(["LFPO_LFMN", "LFMN_LFPO"]), ["ANMOIS", "PAP", "PAP_PAX"]].head(4))

               type  manquants
ANMOIS          str          0
PAP             str          0
PAP1_LABEL      str          0
PAP2_LABEL      str          0
PAP_DIST      int64          0
PAP_PEQ       int64          0
PAP_PAX       int64          0
PAP_FRP     float64          0
PAP_DRT       int64          0
PAP_MVT       int64          0
PAP_PEQ_FS    int64          0
PAP_PAX_FS    int64          0
PAP_FRP_FS  float64          0
lignes intérieures à la métropole : 476 sur 3572
     ANMOIS        PAP  PAP_PAX
94   201701  LFPO_LFMN   155167
324  201703  LFPO_LFMN   171111
623  201705  LFPO_LFMN   200434
781  201706  LFPO_LFMN   208092


### Ce que dit la notice DGAC

- Une ligne correspon à une liaison et un mois, départs et arrivées confondus (pas de doublon par sens).
- Ne sont publiés que les mois avec au moins 10 000 passagers équivalents, au moins 3 compagnies
  et aucune au-dessus de 85 % du trafic (secret statistique).
  Une liaison absente n'a donc pas zéro passager : elle reste NaN.
- On retient `PAP_PAX`, les passagers locaux de la liaison : ce sont les voyageurs qui vont réellement d'une ville à l'autre, c'est-à-dire ceux qui pourraient prendre le train à la place de l'avion.

voici l'url de la notice: https://static.data.gouv.fr/resources/trafic-aerien-commercial-mensuel-francais-par-paire-daeroports-par-sens-depuis-1990/20260903-120427/notice-asp-serie-pap.pdf

In [ ]:
# on garde les liaisons intérieures et on ajoute l'année
dom = pap[interieur].copy()
dom["annee"] = dom["ANMOIS"].str[:4]   # ANMOIS vaut par exemple 201702, les quatre premiers caractères sont donc l'année, ce que l'on cherche à isoler et à compter

# pour chaque liaison et chaque année : nombre de mois publiés et passagers
par_liaison = dom.groupby(["PAP", "annee"]).agg(
    mois_publies=("ANMOIS", "count"),
    passagers=("PAP_PAX", "sum"),
).reset_index()   # .reset_index() remet PAP et annee en colonnes normales, pour avoir un tableau simple.

print("liaisons par année :")
print(par_liaison["annee"].value_counts())
print()
print("nombre de mois publiés (combien de liaisons dans chaque cas) :")
print(par_liaison.groupby("annee")["mois_publies"].value_counts().sort_index())

liaisons par année :
annee
2017    41
2023    39
Name: count, dtype: int64

nombre de mois publiés (combien de liaisons dans chaque cas) :
annee  mois_publies
2017   1                3
       2                4
       3                3
       4                3
       5                4
       6                3
       7                5
       8                2
       9                2
       10               1
       11               1
       12              10
2023   1                9
       2                6
       3                5
       4                2
       5                2
       6                2
       7                1
       8                3
       9                2
       10               1
       11               1
       12               5
Name: count, dtype: int64


In [ ]:
# numéro du mois (01 à 12), pour comparer un mois avec le même mois
dom["mois"] = dom["ANMOIS"].str[4:]

# une ligne par liaison et par mois, une colonne par année
cote_a_cote = dom.pivot_table(index=["PAP", "mois"], columns="annee", values="PAP_PAX")

# mois publiés à la fois en 2017 et en 2023
communs = cote_a_cote.dropna()

# nombre de mois communs pour chaque liaison
mois_communs = communs.groupby("PAP").size()
print("liaisons avec au moins un mois commun :", len(mois_communs))
print()
print("nombre de mois communs (combien de liaisons dans chaque cas) :")
print(mois_communs.value_counts().sort_index())

liaisons avec au moins un mois commun : 24

nombre de mois communs (combien de liaisons dans chaque cas) :
1     4
2     3
3     1
4     2
5     2
6     2
7     1
8     1
9     2
10    1
11    1
12    4
Name: count, dtype: int64


### Problèmes liés aux traitements des données et au nettoyage de la base

#### 1. Le fichier contient tout le trafic, pas seulement les vols intérieurs
Sur les 3 572 observations, seulement 476 relient des aéroports au sein de la métropole. En effet, la DGAC publie tout le trafic qui touche la France (dont vols internationaux, avec les DOM et aussi en métropole). Il y a donc une nécessité de filtrer les vols intra métropoles des autres. Pour cela, il faut garder uniquement les liaisons dont les deux codes OACI commencent par `LF` (France métropolitaine et Corse).

#### 2. A cause du secret statistique, il manque plusieurs mois de données pour certains aéroports
Par exemple, Orly – Nice n'a pas de données en février ni en avril 2017, alors que la liaison dépasse largement 10 000 passagers par mois. la DGAC ne publie un mois que s'il y a au moins 3 compagnies
  et qu'aucune ne dépasse 85 % du trafic (notice de la série PAP). Additionner les mois publiés sous-estime le trafic annuel, et fausse la comparaison 2017 / 2023 si le nombre de mois diffère.
Comme solution, un peu pis à lait, on a décider de ne garder que les mois publiés entre les années pour comparer 2017 et 2023, on ne garde, pour chaque liaison, que les mois publiés
  les deux années (par exemple mars 2017 et mars 2023). Un mois caché par le secret statistique
  est ainsi écarté les deux années, et l'effet des saisons s'annule.
  24 liaisons ont au moins un mois en commun (4 en ont 12, 7 en ont seulement 1 ou 2).
  On garde le nombre de mois communs dans une colonne `mois_communs` pour signaler les
  évolutions fragiles. Les liaisons publiées une seule année servent à décrire cette année-là,
  mais pas à calculer une évolution.

### Ce qui n'est pas un problème
- Aucune valeur manquante, codes déjà en texte.
- Chaque liaison n'apparaît que dans un sens, les deux sens étant additionnés par la DGAC
  (vérifié sur Orly – Nice) : rien à regrouper.

In [ ]:
from src.nettoyage import clean_dgac

liaisons = clean_dgac(pap, 2017, 2023)
print(liaisons.shape)
print("liaisons avec au moins un mois commun :", (liaisons["mois_communs"] > 0).sum())
liaisons[liaisons["code_liaison"] == "LFPO_LFMN"].T

(52, 13)
liaisons avec au moins un mois commun : 24


,43
code_liaison,LFPO_LFMN
code_oaci_1,LFPO
code_oaci_2,LFMN
nom_aeroport_1,PARIS-ORLY
nom_aeroport_2,NICE-COTE D'AZUR
dist_km,674
mois_publies_2017,8
passagers_2017,1448041.0
mois_publies_2023,8
passagers_2023,1098774.0


In [ ]:
# sauvegarde de la table propre
liaisons.to_parquet("data/clean/liaisons_aeriennes.parquet")
print("manquants par colonne :")
print(liaisons.isna().sum())

manquants par colonne :
code_liaison               0
code_oaci_1                0
code_oaci_2                0
nom_aeroport_1             0
nom_aeroport_2             0
dist_km                    0
mois_publies_2017          0
passagers_2017            11
mois_publies_2023          0
passagers_2023            13
mois_communs               0
passagers_communs_2017    28
passagers_communs_2023    28
dtype: int64


Les valeurs manquantes sont celles attendues. En effet, il s'agit de toutes les liaisons non publiées edans la base de données (11 en 2017, 13 en 2023 et 28 sans mois commun). Pour ne pas fausser les données les données manquantes n'ont pas été remplacées par des 0, elles sont donc écartées de l'analyse, cela reste une grosse limite de notre analyse à cause de cette base lacunaire. 

### Bilan

Fichier propre : `data/clean/liaisons_aeriennes.parquet`

- 52 liaisons intérieures à la métropole publiées en 2017 ou en 2023 (41 en 2017, 39 en 2023).
- 24 ont au moins un mois publié les deux années : ce sont elles qui servent à mesurer l'évolution.
- Les passagers d'une année sont un minimum, à cause des mois cachés par le secret statistique.
  Une liaison non publiée reste NaN, jamais 0.

**Exemple : Orly – Nice.** La liaison est publiée 8 mois en 2017 et 8 mois en 2023, dont 5 mois
en commun. Sur ces 5 mois, le trafic passe de 916 663 à 688 391 passagers, soit environ -25 %.
Comparer les totaux annuels (1 448 041 contre 1 098 774) n'aurait pas été juste, car les mois
publiés ne sont pas tous les mêmes d'une année à l'autre.

## Exploration des facteurs d'émission (ADEME, Base Carbone)

Source : API Base Carbone de l'ADEME (data.ademe.fr).
Facteurs retenus : avion (43743), TGV (43256), Intercités (43272), TER (43255), voiture (43791).
Avion : 101 à 220 sièges, vols de 500 à 1 000 km, avec traînées (un seul facteur pour toutes
les liaisons : limite à signaler).

In [ ]:
from src.collecte import get_base_carbone

# les identifiants ADEME sont les clés du dictionnaire facteurs_ademe
bc = get_base_carbone(config["urls"]["base_carbone"], list(config["facteurs_ademe"]))

# les colonnes qui décrivent chaque facteur, une ligne par facteur
colonnes = ["Identifiant_de_l'élément", "Nom_base_français", "Nom_attribut_français",
            "Nom_frontière_français", "Total_poste_non_décomposé", "Unité_français",
            "Statut_de_l'élément", "Localisation_géographique", "Date_de_modification"]
bc[colonnes]

,Identifiant_de_l'élément,Nom_base_français,Nom_attribut_français,Nom_frontière_français,Total_poste_non_décomposé,Unité_français,Statut_de_l'élément,Localisation_géographique,Date_de_modification
0,43743,Avion passagers,"101-220 sièges, 500-1000 kms, 2023",AVEC trainées,0.22500,kgCO2e/passager.km,Valide générique,France continentale,2025-04-24
1,43272,Intercités,2022,NaN,0.00750,kgCO2e/passager.km,Valide générique,France continentale,2024-07-04
2,43255,TER,2022,Traction moyenne,0.02770,kgCO2e/passager.km,Valide générique,France continentale,2023-09-27
3,43256,TGV,2022,NaN,0.00293,kgCO2e/passager.km,Valide générique,France continentale,2023-09-27
4,43791,Voiture,Motorisation moyenne,2023,0.25600,kgCO2e/km,Valide générique,France continentale,2025-04-24


### Constats

On utilise 5 facteurs d'émission carbone, qui correspond à chaque mode de transport, sur 49 colonnes (décrivant le gaz, la qualité de la données). Ces 49 colonnes vont être simplifiées pour ne pas compléxifier inutilement l'analyse centrale. Tous les facteurs ont ici le statut "Valide générique", ce qui suit les recommandations de l'ADEME.
Il y a une attention particulière qui doit être portée sur les unités, elles ne sont pas homogènes entre elles. Pour l'avion le TGV, les Intercités et le TER, l'unité de référence est le kgCO2e par passager.km 
La voiture utilise le kgCO2e oar véhicule.km. POur les comparer, il faut diviser par le nombre d'occupants par voiture. Il s'agit d'une hypothèse supplémentaire. Il y a un problème d'ajustement au niveau des années aussi. Le train c'est 2022, pour l'avion et la voiture c'est 2023. Il s'agit des dernières valeurs publiées par l'ADEME. Ce léger décalage ne change pas drastiquement les interprétations et les résultats à observer, car il s'agit surtout de facteurs technologiques et tendanciels. Ils évoluent donc lentement. I reste tout de même essentiel de mentionner ce décalage. 

### Ce que fera le nettoyage (`clean_base_carbone`)

1. Garder une ligne par mode, qui sera composée du mode, de l'identifiant ADEME, de la valeur, de l'unité ;
2. Donner des noms plus courts aux valeurs de la variable mode : `avion`, `tgv`, `intercites`, `ter`, `voiture` ;
3. vérifier par des `assert`qu'il y a bien un seul facteur par mode et aucune valeur manquante.

In [ ]:
from src.nettoyage import clean_base_carbone

modes = {"43743": "avion", "43256": "tgv", "43272": "intercites", "43255": "ter", "43791": "voiture"}
facteurs = clean_base_carbone(bc, modes)
facteurs

,mode,code_ademe,co2_kg_km,unite
0,avion,43743,0.22500,kgCO2e/passager.km
1,intercites,43272,0.00750,kgCO2e/passager.km
2,ter,43255,0.02770,kgCO2e/passager.km
3,tgv,43256,0.00293,kgCO2e/passager.km
4,voiture,43791,0.25600,kgCO2e/km


In [ ]:
# sauvegarde de la table propre
facteurs.to_parquet("data/clean/facteurs_emission.parquet")
print("modes :", facteurs["mode"].tolist())

modes : ['avion', 'intercites', 'ter', 'tgv', 'voiture']


### Bilan

Fichier propre : `data/clean/facteurs_emission.parquet`

**Limites:**
En raison d'une contrainte de temps, on ne considère qu'un facteur pour un avion pour les liaisons, alors que les vols sont de distance plus courtes que les autres. Or, un avion qui parcours des distances plus courtes consomme plus au km. Les mêmes facteurs sont appliqués en 2017 et en 2023. Il s'agit de comparer à technologie égale. Cela peut être une limite, si on cherche à comparer les émissions sur le long terme et à comptabiliser leur diminution notamment. 